<a href="https://colab.research.google.com/github/kumarsirish/RAG_ASSIGNMENT_LEGAL/blob/main/RAG_Assg_Legal_Documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Extracting Information from Legal Documents Using RAG**

## **Objective**

The main objective of this assignment is to process and analyse a collection text files containing legal agreements (e.g., NDAs) to prepare them for implementing a **Retrieval-Augmented Generation (RAG)** system. This involves:

* Understand the Cleaned Data : Gain a comprehensive understanding of the structure, content, and context of the cleaned dataset.
* Perform Exploratory Analysis : Conduct bivariate and multivariate analyses to uncover relationships and trends within the cleaned data.
* Create Visualisations : Develop meaningful visualisations to support the analysis and make findings interpretable.
* Derive Insights and Conclusions : Extract valuable insights from the cleaned data and provide clear, actionable conclusions.
* Document the Process : Provide a detailed description of the data, its attributes, and the steps taken during the analysis for reproducibility and clarity.

The ultimate goal is to transform the raw text data into a clean, structured, and analysable format that can be effectively used to build and train a RAG system for tasks like information retrieval, question-answering, and knowledge extraction related to legal agreements.

### **Business Value**  


The project aims to leverage RAG to enhance legal document processing for businesses, law firms, and regulatory bodies. The key business objectives include:

* Faster Legal Research: <br> Reduce the time lawyers and compliance officers spend searching for relevant case laws, precedents, statutes, or contract clauses.
* Improved Contract Analysis: <br> Automatically extract key terms, obligations, and risks from lengthy contracts.
* Regulatory Compliance Monitoring: <br> Help businesses stay updated with legal and regulatory changes by retrieving relevant legal updates.
* Enhanced Decision-Making: <br> Provide accurate and context-aware legal insights to assist in risk assessment and legal strategy.


**Use Cases**
* Legal Chatbots
* Contract Review Automation
* Tracking Regulatory Changes and Compliance Monitoring
* Case Law Analysis of past judgments
* Due Diligence & Risk Assessment

## **1. Data Loading, Preparation and Analysis** <font color=red> [20 marks] </font><br>

### **1.1 Data Understanding**

The dataset contains legal documents and contracts collected from various sources. The documents are present as text files (`.txt`) in the *corpus* folder.

There are four types of documents in the *courpus* folder, divided into four subfolders.
- `contractnli`: contains various non-disclosure and confidentiality agreements
- `cuad`: contains contracts with annotated legal clauses
- `maud`: contains various merger/acquisition contracts and agreements
- `privacy_qa`: a question-answering dataset containing privacy policies

The dataset also contains evaluation files in JSON format in the *benchmark* folder. The files contain the questions and their answers, along with sources. For the above folders, there is a `json` file: `contractnli.json`, `cuad.json`, `maud.json`. The file structure is as follows:

```
{
    "tests": [
        {
            "query": <question1>,
            "snippets": [{
                    "file_path": <source_file1>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 1>
                },
                {
                    "file_path": <source_file2>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 2>
                }, ....
            ]
        },
        {
            "query": <question2>,
            "snippets": [{<answer context for que 2>}]
        },
        ... <more queries>
    ]
}
```

### **1.2 Load and Preprocess the data** <font color=red> [5 marks] </font><br>

#### Loading libraries

In [1]:
## The following libraries might be useful
#%pip install -q langchain-openai
#%pip install -U -q langchain-community
#%pip install -U -q langchain-chroma
#%pip install -U -q datasets pyarrow pandas
#%pip install -U -q ragas
#%pip install -U -q rouge_score
#%pip install -q sentence-transformers qdrant-client
!rm -f requirements.txt
! wget -O requirements.txt https://raw.githubusercontent.com/kumarsirish/RAG_ASSIGNMENT_LEGAL/main/requirements.txt

%pip install -q -r requirements.txt


--2026-06-20 09:25:57--  https://raw.githubusercontent.com/kumarsirish/RAG_ASSIGNMENT_LEGAL/main/requirements.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 261 [text/plain]
Saving to: ‘requirements.txt’

requirements.txt    100%[===================>]     261  --.-KB/s    in 0s      

2026-06-20 09:25:58 (6.09 MB/s) - ‘requirements.txt’ saved [261/261]



In [2]:
#!pip freeze
# Import essential libraries



#### **1.2.1** <font color=red> [3 marks] </font>
Load all `.txt` files from the folders.

You can utilise document loaders from the options provided by the LangChain community.

Optionally, you can also read the files manually, while ensuring proper handling of encoding issues (e.g., utf-8, latin1). In such case, also store the file content along with metadata (e.g., file name, directory path) for traceability.

In [3]:
import os
import json
from langchain_community.document_loaders import DirectoryLoader, TextLoader

if not os.path.exists("benchmarks") or not os.path.exists("corpus"):
  !rm -rf corpus benchmarks benchmark corpus.zip benchmark.zip
  !wget -q -O benchmark.zip https://github.com/kumarsirish/RAG_ASSIGNMENT_LEGAL/raw/main/benchmark.zip
  !unzip -q -o corpus.zip -d .
  !unzip -q -o benchmark.zip -d .

# Some archives unpack as benchmark/; normalize to benchmarks/
if os.path.exists("benchmark") and not os.path.exists("benchmarks"):
    os.rename("benchmark", "benchmarks")

corpus_path = "./corpus"
benchmark_path = "./benchmarks"

# Load corpus .txt files as LangChain Documents
documents = DirectoryLoader(
    corpus_path,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"autodetect_encoding": True},
    silent_errors=True,
).load()

# Optional metadata cleanup
for d in documents:
    d.metadata["file_name"] = os.path.basename(d.metadata.get("source", ""))

# Load benchmark json files
benchmarks = {}
for file_name in os.listdir(benchmark_path):
    if file_name.endswith(".json"):
        file_path = os.path.join(benchmark_path, file_name)
        with open(file_path, "r", encoding="utf-8") as f:
            benchmarks[file_name.replace(".json", "")] = json.load(f)

print("Docs loaded:", len(documents))
print("Benchmarks loaded:", list(benchmarks.keys()))


/tmp/ipykernel_24344/1552054436.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Docs loaded: 644
Benchmarks loaded: ['maud', 'cuad', 'contractnli']


#### **1.2.2** <font color=red> [2 marks] </font>
Preprocess the text data to remove noise and prepare it for analysis.

Remove special characters, extra whitespace, and irrelevant content such as email and telephone contact info.
Normalise text (e.g., convert to lowercase, remove stop words).
Handle missing or corrupted data by logging errors and skipping problematic files.

In [4]:
# Clean and preprocess the data
# remove emails from the documents
import re
import nltk
nltk.download('stopwords')

#remove emails from the documents
def remove_emails(text):
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    return re.sub(email_pattern, '', text)

# remove phone numbers from the documents
def remove_phone_numbers(text):
    phone_pattern = r'\+?\d[\d -]{8,}\d'
    return re.sub(phone_pattern, '', text)

# remove special characters from the documents
def remove_special_characters(text):
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

#remove extra spaces from the documents
def remove_extra_spaces(text):
    return re.sub(r"[ \t]+", " ", text).strip()

#convert the documents to lowercase
def convert_to_lowercase(text):
    return text.lower()

#remove stop words from the documents using NLTK's stopwords
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
def remove_stop_words(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

# apply the pre-processing functions to the documents
for doc in documents:
    doc.page_content = remove_emails(doc.page_content)
    doc.page_content = remove_phone_numbers(doc.page_content)
    doc.page_content = remove_special_characters(doc.page_content)
    doc.page_content = remove_extra_spaces(doc.page_content)
    doc.page_content = convert_to_lowercase(doc.page_content)
    doc.page_content = remove_stop_words(doc.page_content)

# Check the cleaned document
print(f'Cleaned example document: {documents[0].page_content[:200]}...')






[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Cleaned example document: exhibit 106 attachment erchonia corporation exclusive distributor agreement agreement agreement made erchonia corporation erchonia whose address 650 atlantis rd melbourne florida usa 32904 innerscope ...


### **1.3 Exploratory Data Analysis** <font color=red> [10 marks] </font><br>

#### **1.3.1** <font color=red> [2 marks] </font>
Calculate the average, maximum and minimum document length.

In [5]:
# Calculate the average, maximum and minimum document length.
doc_lengths = []  #count of words in each document

print(f'Number of documents: {len(documents)}')
for doc in documents:
    words = doc.page_content.split()
    length = len(words)
    doc_lengths.append(length)


average_length = sum(doc_lengths) / len(documents)
max_length = max(doc_lengths)
min_length = min(doc_lengths)
print(f'Average document length: {average_length:.0f} words')
print(f'Maximum document length: {max_length} words')
print(f'Minimum document length: {min_length} words')

Number of documents: 644
Average document length: 8505 words
Maximum document length: 84946 words
Minimum document length: 142 words


#### **1.3.2** <font color=red> [4 marks] </font>
Analyse the frequency of occurence of words and find the most and least occuring words.

Find the 20 most common and least common words in the text. Ignore stop words such as articles and prepositions.

In [6]:
# Find frequency of occurence of words
from collections import Counter
import pprint
word_freq = Counter()
for doc in documents:
    words = doc.page_content.split()

    #ignore single letter words
    words = [word for word in words if len(word) > 1]

    #ignore prepositions, conjunctions, articles etc. (stop words)..
    words = [word for word in words if word not in stop_words]

    #shall and may are very common in legal documents, so ignore them as well
    words = [word for word in words if word not in ['shall', 'may']]

    #ignore roman numerals
    words = [word for word in words if not re.match(r'^[ivxlcdm]+$', word)]

    word_freq.update(words)
# Print the 10 most common words and their frequencies
pprint.pprint(word_freq.most_common(20))



[('company', 128586),
 ('agreement', 92848),
 ('section', 65803),
 ('parent', 52482),
 ('party', 44656),
 ('date', 34487),
 ('time', 30923),
 ('merger', 29777),
 ('material', 29408),
 ('subsidiaries', 28986),
 ('applicable', 27366),
 ('including', 25818),
 ('respect', 25402),
 ('stock', 22639),
 ('information', 22524),
 ('parties', 21877),
 ('business', 20669),
 ('prior', 20613),
 ('effective', 18698),
 ('required', 18616)]


#### **1.3.3** <font color=red> [4 marks] </font>
Analyse the similarity of different documents to each other based on TF-IDF vectors.

Transform some documents to TF-IDF vectors and calculate their similarity matrix using a suitable distance function. If contracts contain duplicate or highly similar clauses, similarity calculation can help detect them.

Identify for the first 10 documents and then for 10 random documents. What do you observe?

In [7]:
# Transform the page contents of documents - Modularized approach
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import random

def compute_tfidf_vectors(documents_list):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents_list)
    return vectorizer, tfidf_matrix

def compute_pairwise_similarity(tfidf_matrix):
    similarity_matrix = cosine_similarity(tfidf_matrix)
    return similarity_matrix

def analyze_similarity_pairwise(doc_contents, doc_indices=None):

    vectorizer, tfidf_matrix = compute_tfidf_vectors(doc_contents)

    # Compute pairwise similarity matrix
    similarity_matrix = compute_pairwise_similarity(tfidf_matrix)

    # Create labels for rows/columns
    if doc_indices is None:
        labels = [f'Doc{i}' for i in range(len(doc_contents))]
    else:
        labels = [f'Doc{idx}' for idx in doc_indices]

    # Create DataFrame for better visualization
    sim_df = pd.DataFrame(similarity_matrix, index=labels, columns=labels)

    # Print statistics
    # Get upper triangle values (excluding diagonal). upper_triangle is the array of values from similarity_matrix above the main diagonal (excludes diagonal).
    mask = np.triu(np.ones_like(similarity_matrix, dtype=bool), k=1)
    upper_triangle = similarity_matrix[mask]

    return upper_triangle, sim_df


In [8]:
# create a list of 10 random integers
# Analyze first 10 documents (same subdirectory)
doc_contents_first10 = [doc.page_content for doc in documents[:10]]
similarity, sim_df = analyze_similarity_pairwise(doc_contents_first10)
print("===============FIRST 10 DOCUMENTS (Same Subdirectory)==============")

print(f"\nSimilarity Statistics:")
print(f"  Average similarity: {similarity.mean():.4f}")
print(f"  Max similarity: {similarity.max():.4f}")
print(f"  Min similarity: {similarity.min():.4f}")
print(f"  Std deviation: {similarity.std():.4f}")

print("\nPairwise Similarity Matrix:")
print(sim_df.round(4))

===============FIRST 10 DOCUMENTS (Same Subdirectory)==============

Similarity Statistics:
  Average similarity: 0.1419
  Max similarity: 0.2672
  Min similarity: 0.0588
  Std deviation: 0.0519

Pairwise Similarity Matrix:
        Doc0    Doc1    Doc2    Doc3    Doc4    Doc5    Doc6    Doc7    Doc8  \
Doc0  1.0000  0.0843  0.1211  0.1323  0.1636  0.2672  0.1852  0.1372  0.1404   
Doc1  0.0843  1.0000  0.0681  0.0871  0.1052  0.0588  0.0863  0.0718  0.0706   
Doc2  0.1211  0.0681  1.0000  0.1292  0.1699  0.0716  0.1219  0.1347  0.1237   
Doc3  0.1323  0.0871  0.1292  1.0000  0.2533  0.0874  0.1528  0.1393  0.1474   
Doc4  0.1636  0.1052  0.1699  0.2533  1.0000  0.1097  0.1916  0.1892  0.1888   
Doc5  0.2672  0.0588  0.0716  0.0874  0.1097  1.0000  0.1135  0.0839  0.0940   
Doc6  0.1852  0.0863  0.1219  0.1528  0.1916  0.1135  1.0000  0.1355  0.1489   
Doc7  0.1372  0.0718  0.1347  0.1393  0.1892  0.0839  0.1355  1.0000  0.1686   
Doc8  0.1404  0.0706  0.1237  0.1474  0.1888  0.0940  0.

In [9]:
# Compute similarity scores for 10 random documents

# Analyze 10 random documents (mixed subdirectories)
random_indices = random.sample(range(len(documents)), 10)
doc_contents_random = [documents[idx].page_content for idx in random_indices]
similarity, sim_df = analyze_similarity_pairwise(doc_contents_random, random_indices)

print("===============10 RANDOM DOCUMENTS (Mixed Subdirectories)==============")
print(f"\nSimilarity Statistics:")
print(f"  Average similarity: {similarity.mean():.4f}")
print(f"  Max similarity: {similarity.max():.4f}")
print(f"  Min similarity: {similarity.min():.4f}")
print(f"  Std deviation: {similarity.std():.4f}")

print("\nPairwise Similarity Matrix:")
print(sim_df.round(4))

===============10 RANDOM DOCUMENTS (Mixed Subdirectories)==============

Similarity Statistics:
  Average similarity: 0.2025
  Max similarity: 0.8871
  Min similarity: 0.0501
  Std deviation: 0.1864

Pairwise Similarity Matrix:
         Doc37  Doc473  Doc402  Doc267  Doc338  Doc489  Doc520  Doc116  \
Doc37   1.0000  0.1374  0.0892  0.0501  0.0787  0.1129  0.1096  0.1084   
Doc473  0.1374  1.0000  0.2442  0.1531  0.2231  0.8871  0.8693  0.2699   
Doc402  0.0892  0.2442  1.0000  0.0990  0.1532  0.2163  0.1991  0.2070   
Doc267  0.0501  0.1531  0.0990  1.0000  0.1107  0.1351  0.1226  0.1639   
Doc338  0.0787  0.2231  0.1532  0.1107  1.0000  0.1992  0.1833  0.2001   
Doc489  0.1129  0.8871  0.2163  0.1351  0.1992  1.0000  0.8252  0.2324   
Doc520  0.1096  0.8693  0.1991  0.1226  0.1833  0.8252  1.0000  0.2233   
Doc116  0.1084  0.2699  0.2070  0.1639  0.2001  0.2324  0.2233  1.0000   
Doc443  0.0596  0.2861  0.1040  0.0664  0.1042  0.2499  0.2664  0.1355   
Doc636  0.0651  0.1949  0.1158  

### Observation
The first 10 documents show higher similarity because they likely contain significant amount of shared vocabulary or repeated legal boilerplate, while the random documents are more diverse and have lower similarity.

### **1.4 Document Creation and Chunking** <font color=red> [5 marks] </font><br>

#### **1.4.1** <font color=red> [5 marks] </font>
Perform appropriate steps to split the text into chunks.

In [10]:
# Process files and generate chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

#LArger chunks for the legal documents, as they are usually long and complex.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = []
for doc in documents:
    doc_chunks = text_splitter.split_text(doc.page_content)
    chunks.extend(doc_chunks)
print(f'Number of chunks generated: {len(chunks)}')
print(f'Example chunk: {chunks[0][:200]}...')

Number of chunks generated: 55348
Example chunk: exhibit 106 attachment erchonia corporation exclusive distributor agreement agreement agreement made erchonia corporation erchonia whose address 650 atlantis rd melbourne florida usa 32904 innerscope ...


## **2. Vector Database and RAG Chain Creation** <font color=red> [15 marks] </font><br>

### **2.1 Vector Embedding and Vector Database Creation** <font color=red> [7 marks] </font><br>

#### **2.1.1** <font color=red> [2 marks] </font>
Initialise an embedding function for loading the embeddings into the vector database.

Initialise a function to transform the text to vectors using an embedding model. You can also use this function to transform during vector DB creation itself.

In [11]:
# Fetch your API Key as an environment variable (or load it directly if variable naming is conventional)
#QDRANT_HOST = os.getenv("QDRANT_HOST")
#QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
from google.colab import userdata
HF_TOKEN=userdata.get('HF_TOKEN')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GOOGLE_API_KEY = GEMINI_API_KEY

#HF_TOKEN = os.getenv("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN  # Set the Hugging Face token in the environment

if not HF_TOKEN:
    raise ValueError("Please set the  HF_TOKEN environment variables.")

In [12]:
# Initialise an embedding function

from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

model_name = "all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("GPU not available, using CPU. This may be slow for large datasets.")
else:
    print("GPU detected, using CUDA for embeddings.")

embedding_function = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": device}

)

print("Loaded:", model_name)
print("Device:", device)


GPU detected, using CUDA for embeddings.


/tmp/ipykernel_24344/2201936835.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded: all-MiniLM-L6-v2
Device: cuda


#### **2.1.2** <font color=red> [5 marks] </font>
Load the embeddings to a vector database.

Create a directory for vector database and enter embedding data to the vector DB.

In [13]:
import chromadb
from langchain_community.vectorstores import Chroma
import os

# Define the directory for ChromaDB persistence
db_directory = "./chroma_db"

# Check if the directory exists
if not os.path.exists(db_directory):
    print(f"ChromaDB directory '{db_directory}' not found. Creating and populating...")
    # Create a new ChromaDB and populate it
    vector_store = Chroma.from_texts(
        texts=chunks,
        embedding=embedding_function,
        persist_directory=db_directory
    )
    print(f"Added {len(chunks)} chunks to ChromaDB.")
else:
    print(f"ChromaDB directory '{db_directory}' already exists. Loading existing store...")
    # Load the existing ChromaDB collection
    vector_store = Chroma(persist_directory=db_directory, embedding_function=embedding_function)
    # Check if the loaded collection is empty and populate if needed
    if vector_store._collection.count() == 0:
        print("Loaded collection is empty, remove chroma_db directory and re-run to populate with chunks.")




ChromaDB directory './chroma_db' already exists. Loading existing store...


/tmp/ipykernel_24344/2264239460.py:21: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(persist_directory=db_directory, embedding_function=embedding_function)


### **2.2 Create RAG Chain** <font color=red> [8 marks] </font><br>

#### **2.2.1** <font color=red> [5 marks] </font>
Form the complete RAG pipeline.

You can either create a chain or directly the pipeline

In [14]:
%pip install -U -q langchain-google-genai

In [17]:
import torch
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from google.colab import userdata
import os

try:
    # Initialize Gemini LLM if not already done
    if 'llm' not in globals():
        print("Loading Gemini model: gemini-2.5-flash...")

        # Ensure GOOGLE_API_KEY is available
        # It's expected to be set in cf364c7e, but re-checking for robustness
        current_google_api_key = os.getenv('GOOGLE_API_KEY')
        if not current_google_api_key:
            # If not in env, try to get from userdata directly
            current_google_api_key = userdata.get('GOOGLE_API_KEY')
            if not current_google_api_key:
                raise ValueError("GOOGLE_API_KEY not found in Colab secrets or environment. Please set it.")
            os.environ["GOOGLE_API_KEY"] = current_google_api_key # Set for future calls

        genai.configure(api_key=current_google_api_key)

        llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            temperature=0,
            google_api_key=current_google_api_key # Explicitly pass the API key
        )
        print("Gemini LLM loaded successfully.")

    # Set up retriever
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    # Prompt Template
    template = """You are an expert legal assistant. Answer the user's question using only the provided context below.
If you do not know the answer or if it's not explicitly mentioned in the context, say that you don't know. Do not make things up.

Context:
{context}

Question:
{question}

Answer:"""

    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        formatted_docs = "\n\n".join(doc.page_content for doc in docs)
        print("*********************RETRIEVED DOCS ****************")
        print(f"Retrieved docs: {formatted_docs}")
        print("************************************ ****************")

        return formatted_docs

    def format_docs_for_qa(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # Create the RAG chain using Gemini LLM
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    print("RAG chain successfully initialized with Gemini model")

except Exception as e:
    print(f"An error occurred: {e}")
    print("Ensure that 'vector_store' is defined in previous cells and GOOGLE_API_KEY is set in secrets.")

Loading Gemini model: gemini-2.5-flash...
Gemini LLM loaded successfully.
RAG chain successfully initialized with Gemini model


#### **2.2.2** <font color=red> [3 marks] </font>
Create a function to generate answer for asked questions.

Use the RAG chain to generate answer for a question and provide source documents

In [18]:
# Create a function for question answering
question = "What are the termination obligations under the NDA?"
response = rag_chain.invoke(question)

print(response)



*********************RETRIEVED DOCS ****************
Retrieved docs: termination h event valid termination agreement pursuant section 81 termination agreement shall forthwith become void shall liability obligation part party hereto except provided section 66c access information confidentiality section 68 publicity expense reimbursement indemnification provisions section 611g parent financing section 82 section 83 expenses article ix shall survive termination accordance terms conditions provided subject limitations set forth section 82e section 82f section 82g nothing herein shall relieve company liability damages resulting companys willful breach prior termination party hereto parties acknowledge agree nothing section 82 shall deemed affect right specific performance accordance terms conditions set forth section 912 event agreement validly terminated company pursuant section 81dii superior proposal parent pursuant section 81eii change recommendation company shall pay company terminatio

In [19]:
# Example question
question ="Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?"
response = rag_chain.invoke(question)
print(response)


*********************RETRIEVED DOCS ****************
Retrieved docs: party agreement 4 confidential information shall also include shall limited information disclosed disclosing party writing marked confidential time disclosure b information disclosed disclosing party orally slated confidential time disclosure c information disclosed manner designated writing confidential information time disclosure notwithstanding subclauses ab c definition information whose nature makes obvious confidential e confidential information shall include information whichis time disclosure publicly known f becomes later date publicly available otherwise wrongful act negligence breach agreement receiving party g receiving party demonstrate written records possession known receiving party receipt agreement previously acquired obligation confidentiality h legitimately obtained time receiving party third party without restrictions respect disclosure use receiving party demonstrate satisfaction disclosing party 

## **3. RAG Evaluation** <font color=red> [10 marks] </font><br>

### **3.1 Evaluation and Inference** <font color=red> [10 marks] </font><br>

#### **3.1.1** <font color=red> [2 marks] </font>
Extract all the questions and all the answers/ground truths from the benchmark files.

Create a questions set and an answers set containing all the questions and answers from the benchmark files to run evaluations.

In [20]:
# Create a question set by taking all the questions from the benchmark data
# Also create a ground truth/answer set
import json
import os
import random

benchmark_dir = "./benchmarks"  # adjust if your path differs
benchmark_files = ["contractnli.json", "cuad.json", "maud.json"]

all_questions = []
all_answers = []

for filename in benchmark_files:
    filepath = os.path.join(benchmark_dir, filename)
    if not os.path.exists(filepath):
        print(f"Warning: {filepath} not found, skipping.")
        continue

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    for test in data.get("tests", []):
        query = test.get("query", "").strip()
        snippets = test.get("snippets", [])

        # Collect all answer texts for this question as a combined ground truth
        ground_truth_parts = [s.get("answer", "").strip() for s in snippets if s.get("answer", "").strip()]
        ground_truth = " ".join(ground_truth_parts)

        if query and ground_truth:
            all_questions.append(query)
            all_answers.append(ground_truth)

print(f"Total questions loaded: {len(all_questions)}")
print(f"Total ground truths loaded: {len(all_answers)}")
print(f"\nSample question: {all_questions[0]}")
print(f"Sample ground truth: {all_answers[0][:300]}...")


Total questions loaded: 6695
Total ground truths loaded: 6695

Sample question: Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?
Sample ground truth: Any and all proprietary rights, including but not limited to rights to and in inventions, patent rights, utility models, copyrights, trademarks and trade secrets, in and to any Confidential Information shall be and remain with the Participants respectively, and Mentor shall not have any right, licen...


#### **3.1.2** <font color=red> [5 marks] </font>
Create a function to evaluate the generated answers and retrieved contexts.

Evaluate the responses with *Ragas*. Additionally check the retrieval quality using 2 retrieval-driven metrics.

In [21]:
import sys
import types

# Compatibility shim: newer langchain_community dropped VertexAI modules
# but RAGAS may still try to import them at module load time.
for _mod_name, _cls_name in [
    ('langchain_community.chat_models.vertexai', 'ChatVertexAI'),
    ('langchain_community.llms.vertexai', 'VertexAI'),
]:
    if _mod_name not in sys.modules:
        _fake = types.ModuleType(_mod_name)
        setattr(_fake, _cls_name, type(_cls_name, (), {}))
        sys.modules[_mod_name] = _fake

import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


def evaluate_rag_pipeline(questions, ground_truths, batch_size=2):
    """
    Evaluates the RAG pipeline using RAGAS metrics with HuggingFace LLM.
    Processes questions in batches for GPU efficiency.
    """
    eval_data = {
        "question": [],
        "answer": [],
        "contexts": [],
        "ground_truth": [],
    }

    print(f"Processing {len(questions)} samples in batches of {batch_size}...")

    # Process in batches for GPU efficiency
    for i in range(0, len(questions), batch_size):
        batch_questions = questions[i:i+batch_size]
        batch_ground_truths = ground_truths[i:i+batch_size]

        # Batch retrieve contexts
        batch_contexts = [retriever.invoke(q) for q in batch_questions]

        # Batch generate answers
        batch_answers = [rag_chain.invoke(q) for q in batch_questions]

        # Add to eval data
        for q, answer, contexts, gt in zip(batch_questions, batch_answers, batch_contexts, batch_ground_truths):
            eval_data["question"].append(q)
            eval_data["answer"].append(answer)
            eval_data["contexts"].append([d.page_content for d in contexts])
            eval_data["ground_truth"].append(gt)

    ds = Dataset.from_dict(eval_data)

    # Wrap HuggingFace LLM and embeddings for RAGAS
    ragas_llm = LangchainLLMWrapper(llm)
    ragas_embeddings = LangchainEmbeddingsWrapper(embedding_function)

    # Note: metrics are already instances, don't call them with ()
    results = evaluate(
        ds,
        metrics=[faithfulness, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=ragas_embeddings
    )
    return results, eval_data


/tmp/ipykernel_24344/3992173045.py:18: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall, context_precision
/tmp/ipykernel_24344/3992173045.py:18: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall, context_precision
/tmp/ipykernel_24344/3992173045.py:18: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, context_recall, context_precis

#### **3.1.3** <font color=red> [3 marks] </font>
Draw inferences by evaluating answers to questions.

To save time and computing power, you can just run the evaluation on 10 randomly sampled questions.

In [23]:
import random

# Select 10 random samples to save time and credits
random_samples=10
sample_indices = random.sample(range(len(all_questions)), random_samples)
sample_questions = [all_questions[i] for i in sample_indices]
sample_ground_truths = [all_answers[i] for i in sample_indices]

# Run the evaluation using the function defined in 3.1.2 with batch processing
results, eval_data = evaluate_rag_pipeline(sample_questions, sample_ground_truths, batch_size=4)

# Display results
print("\nEvaluation Results:")
print(results.to_pandas())


Processing 10 samples in batches of 4...
*********************RETRIEVED DOCS ****************
Retrieved docs: agreement shall assignable either party without prior written consent party 13 entire agreement agreement represents complete entire understanding parties regarding subject matter hereof supersedes prior negotiations representations agreements either written oral regarding subject matter agreement shall considered accepted approved otherwise effective signed appropriate parties bravatek technologies inc fazync llc name thomas cellucci name devon jones title ceo title manager date january 10 2018 date january 10 2018

development obligations oil gas leases xviii contract company related party transaction 41 xix agreement contains favored nation favored customer provision call put option preferential right rights first last offer negotiation refusal case contained agreement provision solely benefit company subsidiaries b customary royalty pricing provisions oil gas leases c custo

/tmp/ipykernel_24344/3992173045.py:58: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
/tmp/ipykernel_24344/3992173045.py:59: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embedding_function)


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]


Evaluation Results:
                                          user_input  \
0  Consider the Joint Development Agreement betwe...   
1  Consider the Transportation Services Agreement...   
2  Consider the Acquisition Agreement between Par...   
3  Consider the Sponsored Research and License Ag...   
4  Consider the Content License, Marketing, and S...   
5  Consider the Distributor Agreement between Ven...   
6  Consider eHandshake's Non-Disclosure Agreement...   
7  Consider the Acquisition Agreement between Par...   
8  Consider the Non-Disclosure Agreement between ...   
9  Consider the Strategic Alliance Agreement betw...   

                                  retrieved_contexts  \
0  [agreement shall assignable either party witho...   
1  [exhibit 103 transportation services agreement...   
2  [section 1114 financing provisions 108 exhibit...   
3  [management term agreement principal investiga...   
4  [content license agreement agreement made 2nd ...   
5  [bundled therefore fee 

## **4. Conclusion** <font color=red> [5 marks] </font><br>

### **4.1 Conclusions and insights** <font color=red> [5 marks] </font><br>

#### **4.1.1** <font color=red> [5 marks] </font>
Conclude with the results here. Include the insights gained about the data, model pipeline, the RAG process and the results obtained.